# Basira — Output Generation & Validation

Generates all committed geospatial outputs:
- NDVI classification
- Terrain hillshade
- VIIRS night lights
- Demand intelligence chart
- Results grid
- Sample Folium map


In [ ]:
# Imports
import os
import sys
import json
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import rasterio
from rasterio.mask import mask as raster_mask
from rasterio.warp import reproject, Resampling
import geopandas as gpd
from shapely.geometry import box, mapping
import pyproj
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Path setup
_THIS_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
if _THIS_DIR not in sys.path:
    sys.path.insert(0, _THIS_DIR)

_ROOT_DIR = os.path.dirname(_THIS_DIR)
if _ROOT_DIR not in sys.path:
    sys.path.insert(0, _ROOT_DIR)

CONFIG_PATH = "config/alquaa.json"
DEMAND_PATH = "outputs/demand_scores_alquaa.json"
OUTPUT_DIR = "outputs"
ASSETS_DIR = "assets"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(ASSETS_DIR, exist_ok=True)

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    cfg = json.load(f)

with open(DEMAND_PATH, "r", encoding="utf-8") as f:
    demand = json.load(f)

BBOX = cfg["bbox"]
B04 = cfg["sentinel2_b04"]
B08 = cfg["sentinel2_b08"]
DEM = cfg["dem_tile"]
VIIRS = cfg["viirs_tile"]
OSM = cfg["osm_geojson"]

In [ ]:
PALETTE = {
    "violet": "#7F00FF",
    "deep_purple": "#2E003E",
    "lavender": "#C8A2C8",
    "plum": "#8E4585",
    "hot_pink": "#FF007F",
    "shocking_pink": "#FF4DCC",
    "bg": "#0D0010",
    "text": "#E8D5F5",
    "grid": "#3D1A5E",
    "success": "#3FB950",
}

CATEGORY_LABELS = {
    "1.1": "Primary Resource\n",
    "1.2": "Home Production\n",
    "1.3": "Micro Manufacturing\n",
    "2.1": "Local Services\n",
    "2.2": "Mobility Services\n",
    "2.3": "Community Services\n",
    "2.4": "Professional Svcs\n",
    "3.1": "Retail & Distrib.\n",
    "3.2": "Social & Cultural\n",
    "3.3": "Tourism & Env.\n",
}

In [ ]:
# Helpers
def bbox_geom_for_raster(bbox, raster_path):
    south, west, north, east = bbox
    with rasterio.open(raster_path) as src:
        raster_crs = src.crs
    if raster_crs.to_epsg() == 4326:
        return mapping(box(west, south, east, north))
    transformer = pyproj.Transformer.from_crs(
        "EPSG:4326", raster_crs.to_string(), always_xy=True
    )
    x0, y0 = transformer.transform(west, south)
    x1, y1 = transformer.transform(east, north)
    return mapping(box(x0, y0, x1, y1))

def read_band(path, bbox):
    geom = bbox_geom_for_raster(bbox, path)
    with rasterio.open(path) as src:
        data, transform = raster_mask(src, [geom], crop=True, nodata=0)
    return data[0].astype(float), transform, None

def apply_style(fig, axes=None):
    fig.patch.set_facecolor(PALETTE["bg"])
    if axes is not None:
        for ax in (axes if hasattr(axes, "__iter__") else [axes]):
            ax.set_facecolor(PALETTE["bg"])
            ax.tick_params(colors=PALETTE["text"], labelsize=8)
            for spine in ax.spines.values():
                spine.set_edgecolor(PALETTE["grid"])

## 1. NDVI Classification

In [ ]:
nir, _, _ = read_band(B08, BBOX)
red, _, _ = read_band(B04, BBOX)

valid = (nir + red) > 0
ndvi = np.where(valid, (nir - red) / (nir + red + 1e-10), np.nan)

classes = np.full(ndvi.shape, np.nan)
classes = np.where((ndvi >= -1) & (ndvi < 0.05), 0, classes)
classes = np.where((ndvi >= 0.05) & (ndvi < 0.15), 1, classes)
classes = np.where((ndvi >= 0.15) & (ndvi < 0.25), 2, classes)
classes = np.where((ndvi >= 0.25) & (ndvi <= 1.0), 3, classes)

cmap = mcolors.ListedColormap(["#C4A35A", "#8E6B3E", "#4E7C4E", "#1A5C1A"])

fig, ax = plt.subplots(figsize=(7, 5.5))
apply_style(fig, ax)

ax.imshow(classes, cmap=cmap, interpolation="nearest")
plt.tight_layout()
plt.show()

## 2. Terrain Hillshade

In [ ]:
dem_arr, _, _ = read_band(DEM, BBOX)
dem_arr = np.where(dem_arr == 0, np.nan, dem_arr)

res = 1
dy, dx = np.gradient(np.nan_to_num(dem_arr), res)

azimuth = np.radians(315)
altitude = np.radians(45)

slope = np.arctan(np.sqrt(dx**2 + dy**2))
aspect = np.arctan2(-dy, dx)

hillshade = (np.sin(altitude) * np.cos(slope) +
             np.cos(altitude) * np.sin(slope) * np.cos(azimuth - aspect))

hillshade = np.clip(hillshade, 0, 1)

fig, ax = plt.subplots(figsize=(7, 5.5))
apply_style(fig, ax)
ax.imshow(hillshade, cmap="gray")
plt.show()

## 3. VIIRS Night Lights

In [ ]:
viirs_arr, _, _ = read_band(VIIRS, BBOX)

valid = viirs_arr > 0
p95 = np.percentile(viirs_arr[valid], 95) if np.any(valid) else 1
viirs_norm = np.clip(viirs_arr / (p95 + 1e-9), 0, 1)

cmap = mcolors.LinearSegmentedColormap.from_list(
    "night", ["#000000", PALETTE["violet"], PALETTE["shocking_pink"], "#FFFFFF"]
)

fig, ax = plt.subplots(figsize=(7, 5.5))
apply_style(fig, ax)
ax.imshow(viirs_norm, cmap=cmap)
plt.show()

## 4. Demand Scores Chart

In [ ]:
cats = sorted(demand.keys())
scores = [demand[c]["demand_score"] for c in cats]

fig, ax = plt.subplots(figsize=(9, 5.5))
apply_style(fig, ax)
ax.barh(range(len(cats)), scores)
plt.show()

## 5. Results Grid

In [ ]:
fig = plt.figure(figsize=(12, 9))
gs = GridSpec(2, 2, figure=fig)

ax = fig.add_subplot(gs[0, 0])
ax.imshow(ndvi)

ax = fig.add_subplot(gs[0, 1])
ax.imshow(hillshade)

ax = fig.add_subplot(gs[1, 0])
ax.imshow(viirs_norm)

ax = fig.add_subplot(gs[1, 1])
ax.bar(range(len(scores)), scores)

plt.show()

## 6. Map Generation (external modules)

In [1]:
from scripts.explainability import generate_full_explanation
from scripts.map_engine import generate_map

best_cat = max(demand.keys(), key=lambda c: demand[c]["demand_score"])
signals = dict(demand[best_cat]["signals"])
signals["demand_score"] = demand[best_cat]["demand_score"]

explanation = generate_full_explanation(signals, best_cat)

generate_map(cfg, best_cat, signals, explanation["ar"], output_path="outputs/basira_map_sample.html")

NameError: name 'demand' is not defined